# P2: Análise espacial, Random Forest e MLP

**Projeto:** previsão do tempo de viagem de ônibus entre paradas consecutivas, linha 415 (Usina–Leblon), Rio de Janeiro, 25/01 a 21/03/2019.

**Responsável:** Pessoa 2.

1. Configuração e carregamento
2. Preparação e split
3. EDA espacial (onde o ônibus é lento?)
4. Features
5. Baseline
6. Random Forest (ensemble por bagging)
7. MLP (rede neural)
8. Comparação e análise do erro por parada
9. ETA até a parada X (a aplicação)
10. Exportar os modelos

> Antes de rodar: **Ambiente de execução → Alterar o tipo de ambiente de execução → T4 GPU** (para a MLP).

## 1. Configuração e carregamento

In [ ]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# Arquivos públicos no Google Drive (dados derivados de datasets públicos do Kaggle).
# Baixados por link com gdown: funciona em qualquer conta, sem montar o Drive.
DRIVE_FILE_IDS = {
    "7-segments/segments_415_v0.parquet": "1tLj2N9EPlnRzXaLENXVkb6k_4zGxi56S",
    "6-kaggle-rio/bus_stops_rio_de_janeiro.csv": "1YxgLxu-7F-0k296rW7SEp2dVPKB6SYll",
}

if IN_COLAB:
    import gdown

    DATA_DIR = Path("/content/dados")
    for relative_path, file_id in DRIVE_FILE_IDS.items():
        target = DATA_DIR / relative_path
        target.parent.mkdir(parents=True, exist_ok=True)
        if not target.exists():
            gdown.download(id=file_id, output=str(target), quiet=False)
    if not Path("/content/onibus-ia").exists():
        !git clone -q https://github.com/figdavi/onibus-ia.git /content/onibus-ia
    else:
        !git -C /content/onibus-ia pull -q
    sys.path.insert(0, "/content/onibus-ia/src")
else:
    # Rodando localmente a partir da pasta notebooks/ do repositório
    DATA_DIR = Path("../dados")
    sys.path.insert(0, "../src")

SEGMENTS_PATH = DATA_DIR / "7-segments" / "segments_415_v0.parquet"
STOPS_PATH = DATA_DIR / "6-kaggle-rio" / "bus_stops_rio_de_janeiro.csv"
for path in (SEGMENTS_PATH, STOPS_PATH):
    assert path.exists(), f"arquivo não encontrado: {path}"
print(SEGMENTS_PATH, STOPS_PATH, sep="\n")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import common

pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (12, 4)

raw = pd.read_parquet(SEGMENTS_PATH)
raw["departure_ts"] = pd.to_datetime(raw["departure_ts"])
print(raw.shape)
raw.head()

## 2. Preparação e split

`prepare()` remove dias com poucos dados, velocidades fora de 3–60 km/h e trechos abaixo de 10 s (regras físicas fixas, sem vazamento).

`split()` é da P1. Enquanto não estiver pronto, usamos um split temporal provisório com as mesmas datas do plano.

In [ ]:
df = common.prepare(raw)
print(df.attrs["prepare_report"])

try:
    train, val, test = common.split(df)
    print("split oficial (P1)")
except NotImplementedError:
    day = pd.to_datetime(df["service_date"])
    train = df[day <= "2019-03-03"]
    val = df[day.between("2019-03-04", "2019-03-07")]
    test = df[day.between("2019-03-11", "2019-03-20")]
    print("split PROVISÓRIO; trocar quando a P1 subir split()")

for name, part in [("treino", train), ("validação", val), ("teste", test)]:
    print(f"{name:10s} {len(part):>8,} trechos  {part['service_date'].min()} a {part['service_date'].max()}")

## 3. EDA espacial

Perguntas:
1. Quais paradas e trechos são mais lentos?
2. Há diferença entre bairros e entre os sentidos?
3. O padrão por parada muda ao longo do dia?
4. Por que o sentido Usina tem ~2x mais trechos que o Leblon?

> A EDA usa só o **treino**, para não olhar o teste antes da hora.

In [ ]:
stops = pd.read_csv(STOPS_PATH, sep="|", dtype={"line_number": str})
stops = stops[stops["line_number"] == "415"].copy()
stops["stop_seq"] = stops.groupby("direction").cumcount() + 1  # trecho k sai da parada k
stops = stops[["direction", "stop_seq", "stop_name", "stop_latitude", "stop_longitude", "district"]]

eda = train.assign(speed_kmh=train["distance_m"] / train["travel_time_s"] * 3.6)
by_stop = (eda.groupby(["direction", "stop_seq"])
              .agg(stop_from=("stop_from", "first"), district=("district_from", "first"),
                   n=("travel_time_s", "size"), median_s=("travel_time_s", "median"),
                   speed_kmh=("speed_kmh", "median"), distance_m=("distance_m", "first"))
              .reset_index())
by_stop.sort_values("speed_kmh").head(10)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
for ax, (direction, part) in zip(axes, by_stop.groupby("direction")):
    ax.bar(part["stop_seq"], part["speed_kmh"])
    ax.set_title(f"Velocidade mediana por trecho, sentido {direction}")
    ax.set_ylabel("km/h")
axes[-1].set_xlabel("stop_seq (posição no trajeto)")
plt.tight_layout()

**Interpretação:** _(escreva aqui quais trechos são gargalos e por quê: corredores, túneis, Centro, Zona Sul)_

In [ ]:
by_district = (eda.groupby(["district_from", "direction"])["speed_kmh"].median()
                  .unstack().sort_values("Usina"))
by_district.plot.barh(figsize=(8, 6), title="Velocidade mediana por bairro de origem do trecho")
plt.xlabel("km/h");

**Interpretação:** _(quais bairros são mais lentos?)_

In [ ]:
heat = eda.assign(hour=eda["departure_ts"].dt.hour).pivot_table(
    index="stop_seq", columns="hour", values="speed_kmh", aggfunc="median")
fig, ax = plt.subplots(figsize=(14, 7))
im = ax.imshow(heat, aspect="auto", cmap="RdYlGn", vmin=5, vmax=35, origin="lower")
ax.set_xticks(range(len(heat.columns)), heat.columns)
ax.set_xlabel("hora"); ax.set_ylabel("stop_seq")
ax.set_title("Velocidade mediana (km/h) por trecho e hora, ambos os sentidos")
fig.colorbar(im, ax=ax);

**Interpretação:** _(os gargalos aparecem só no pico? Isso justifica interações parada × hora, que o Ridge não capta)_

### Mapa dos trechos (cor = velocidade mediana)

In [ ]:
import folium

segments_map = by_stop.merge(stops, on=["direction", "stop_seq"], how="left")
nxt = stops.assign(stop_seq=stops["stop_seq"] - 1)[["direction", "stop_seq", "stop_latitude", "stop_longitude"]]
segments_map = segments_map.merge(nxt, on=["direction", "stop_seq"], suffixes=("", "_to")).dropna()

def color(speed):
    return "#d7191c" if speed < 12 else "#fdae61" if speed < 18 else "#a6d96a" if speed < 25 else "#1a9641"

# Tiles da Esri: os do OpenStreetMap bloqueiam o Colab (403, política de uso)
m = folium.Map(location=[stops["stop_latitude"].mean(), stops["stop_longitude"].mean()], zoom_start=13,
               tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
               attr="Tiles &copy; Esri")
for row in segments_map.itertuples():
    folium.PolyLine([(row.stop_latitude, row.stop_longitude), (row.stop_latitude_to, row.stop_longitude_to)],
                    color=color(row.speed_kmh), weight=6, opacity=0.8,
                    tooltip=f"{row.direction} #{row.stop_seq}: {row.stop_from} ({row.speed_kmh:.1f} km/h)").add_to(m)
m

### Desequilíbrio entre sentidos

O sentido Usina tem ~2x mais trechos que o Leblon. Hipóteses:
- **Real:** a linha é circular e os trajetos têm tamanhos/frequências diferentes.
- **Viés do pipeline:** a detecção de sentido (Viterbi) favorece Usina em ruas compartilhadas, ou a cobertura de GPS é pior num sentido.

In [ ]:
coverage = (eda.groupby(["direction", "stop_seq"]).size().unstack(0)
               .plot(figsize=(14, 4), title="Trechos válidos por posição no trajeto"))
plt.ylabel("nº de trechos")

trips = eda.groupby("direction").agg(trechos=("trip_id", "size"), viagens=("trip_id", "nunique"))
trips["trechos_por_viagem"] = trips["trechos"] / trips["viagens"]
trips

**Conclusão:** _(o desequilíbrio é concentrado em alguns trechos? em ruas compartilhadas pelos dois sentidos? viagens do Leblon são mais fragmentadas?)_

## 4. Features

`common.build_features(df, reference=train)` gera 17 colunas:
- **calendário:** hora (crua, seno e cosseno), dia da semana, fim de semana, Carnaval;
- **trecho:** `stop_seq`, `distance_m`, mediana histórica do trecho (`segment_median_s`, calculada **só no treino**) e o sentido em one-hot;
- **trânsito agora** (só o que já aconteceu na partida): trecho anterior da viagem, mesmo trecho pelo ônibus anterior, média dos 3 últimos ônibus no trecho, intervalo entre ônibus, ritmo da viagem no trecho anterior e ritmo médio da viagem até aqui.

`reference` é sempre o **treino**, também para validação e teste. Passar o teste ali vazaria a resposta para as features.

In [ ]:
X_train, y_train = common.build_features(train, reference=train)
X_val, y_val = common.build_features(val, reference=train)
X_test, y_test = common.build_features(test, reference=train)
print(X_train.shape, X_val.shape, X_test.shape)
X_train.isna().mean().round(3)

## 5. Baseline: mediana por sentido, parada, hora e fim de semana

In [ ]:
results = {}
baseline = common.MedianBaseline().fit(X_train, y_train)
results["Baseline (mediana)"] = common.evaluate(baseline, X_test, y_test)
results["Baseline (mediana)"]

## 6. Random Forest (ensemble por bagging)

Muitas árvores treinadas em amostras bootstrap e subconjuntos de features; a previsão é a média. O `SimpleImputer` preenche os lags vazios (a primeira parada de cada viagem não tem trecho anterior).

In [ ]:
import time
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

rf = make_pipeline(
    SimpleImputer(strategy="median"),
    RandomForestRegressor(n_estimators=150, max_depth=20, min_samples_leaf=5,
                          max_features=0.5, n_jobs=-1, random_state=42),
)
start = time.time()
rf.fit(X_train, y_train)
print(f"treino: {time.time() - start:.0f} s")
results["Random Forest"] = common.evaluate(rf, X_test, y_test)
results["Random Forest"]

In [ ]:
importance = pd.Series(rf[-1].feature_importances_, index=X_train.columns).sort_values()
importance.plot.barh(figsize=(7, 5), title="Importância das features (Random Forest)");

**Interpretação:** _(quais features mais importam? os lags confirmam que o trânsito atual prevê o próximo trecho?)_

## 7. MLP (rede neural)

Rede densa com 2 camadas ocultas (ReLU), saída linear e loss MAE. As entradas são imputadas e padronizadas **com estatísticas do treino**. O early stopping no conjunto de validação evita overfitting.

In [ ]:
import tensorflow as tf
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

print("GPU:", tf.config.list_physical_devices("GPU"))
tf.keras.utils.set_random_seed(42)

prep = make_pipeline(SimpleImputer(strategy="median"), StandardScaler()).fit(X_train)
Xtr, Xva, Xte = (prep.transform(x).astype("float32") for x in (X_train, X_val, X_test))

mlp = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(Xtr.shape[1],)),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.1),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(1),
])
mlp.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mae")
history = mlp.fit(
    Xtr, y_train.to_numpy(), validation_data=(Xva, y_val.to_numpy()),
    epochs=50, batch_size=1024, verbose=2,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)],
)

In [ ]:
pd.DataFrame(history.history).plot(title="Curva de aprendizado da MLP (MAE)")
plt.xlabel("época"); plt.ylabel("MAE (s)")

class KerasWrapper:
    def predict(self, X):
        return mlp.predict(prep.transform(X).astype("float32"), verbose=0).ravel()

results["MLP"] = common.evaluate(KerasWrapper(), X_test, y_test)
results["MLP"]

**Interpretação:** _(a curva de validação parou de cair? houve overfitting? a MLP superou a RF?)_

## 8. Comparação e erro por parada

Referências para ler o MAE:
- **Baseline (mediana histórica):** é a régua principal. Um modelo só vale a pena se ganhar dele.
- **Erro de medição (holdout do pipeline, ~25 s):** é um limite **pessimista** do erro da interpolação, não um piso. O holdout esconde 1 de cada 2 pings e dobra o intervalo entre eles, então o erro real da interpolação, que usa todos os pings, é menor. Por isso um modelo pode ficar abaixo de 25 s.

In [ ]:
table = common.metrics_table(results)
table.to_csv(DATA_DIR / "metricas_p2.csv")
table

In [ ]:
errors = test.assign(
    erro_baseline=(baseline.predict(X_test) - y_test).abs(),
    erro_rf=(rf.predict(X_test) - y_test).abs(),
)
(errors.groupby("stop_seq")[["erro_baseline", "erro_rf"]].mean()
       .plot(figsize=(14, 4), title="MAE por posição no trajeto (teste)"))
plt.ylabel("MAE (s)");

## 9. ETA até a parada X (a aplicação)

Os modelos acima preveem **um trecho**. O passageiro pergunta: *"o ônibus está na parada k agora; quando chega na parada k+h?"*. A resposta é a soma das previsões dos próximos h trechos, feita **só com o que se sabe no instante t0** em que o ônibus está na parada k:
- o 1º trecho usa as features reais (todas conhecidas em t0);
- os seguintes usam a **previsão** do trecho anterior como lag, a hora prevista de passagem e só os ônibus anteriores que já tinham chegado em t0.

A previsão é comparada com a **chegada real** (GPS) em dias de teste que o modelo nunca viu. Todos os modelos usam as mesmas 5.000 origens sorteadas.

In [ ]:
import eta

eta_models = {"Baseline (mediana)": baseline, "Random Forest": rf, "MLP": KerasWrapper()}
eta_table = eta.evaluate_eta(eta_models, test, reference=train)
eta_table.to_csv(DATA_DIR / "eta_p2.csv", index=False)

eta_mae = eta_table.pivot(index="stops_ahead", columns="model", values="MAE_s")
eta_mae.plot(marker="o", figsize=(8, 4), title="Erro do ETA por número de paradas à frente (teste)")
plt.xlabel("paradas à frente"); plt.ylabel("MAE (s)")
eta_table

In [ ]:
# Replay de uma viagem real do teste: previsão feita na origem vs. chegada real
enriched_test = common.enrich(test, train).reset_index(drop=True)
origins = enriched_test.iloc[eta.chains(enriched_test, 10)[:, 0]]
example = origins[origins["departure_ts"].dt.hour == 8].iloc[0]  # troque o filtro para outra viagem

trip = eta.replay(eta_models, test, train, example["trip_id"], int(example["stop_seq"]), horizon=10)
t0 = example["departure_ts"]
minutes = trip.set_index("stop")[["real", *eta_models]].apply(lambda col: (col - t0).dt.total_seconds() / 60)
minutes.plot(marker="o", figsize=(12, 4),
             title=f"Viagem real de {t0:%d/%m %H:%M} ({example['direction']}), previsão feita na parada {example['stop_seq']}")
plt.xticks(range(len(minutes)), minutes.index, rotation=60, ha="right"); plt.ylabel("minutos desde a partida")
minutes.round(1)

**Interpretação:** _(quanto o erro cresce com as paradas à frente? o modelo ainda ganha do baseline a 10 paradas? por quê? Dica: a vantagem dos modelos vem principalmente do "trânsito agora", que só é conhecido para o próximo trecho)_

**Conclusões da P2:**
- _(trechos e bairros gargalo)_
- _(desequilíbrio entre sentidos: real ou viés?)_
- _(RF vs. MLP vs. baseline; onde cada modelo erra mais)_

## 10. Exportar os modelos treinados

O modelo treinado vive na memória do Colab e some quando o ambiente é reiniciado. Esta célula salva os arquivos e baixa para o computador. Para carregar em outro lugar (mesmas versões de scikit-learn e TensorFlow):

```python
rf = joblib.load("rf_415.joblib")
prep = joblib.load("mlp_prep.joblib")
mlp = tf.keras.models.load_model("mlp_415.keras")
```

In [ ]:
import os
import joblib

joblib.dump(rf, "rf_415.joblib", compress=3)  # imputer + 150 árvores
joblib.dump(prep, "mlp_prep.joblib")          # imputer + scaler da MLP
mlp.save("mlp_415.keras")                     # pesos da rede

for name in ["rf_415.joblib", "mlp_prep.joblib", "mlp_415.keras"]:
    print(f"{name}: {os.path.getsize(name) / 1e6:.1f} MB")

if IN_COLAB:
    from google.colab import files
    for name in ["rf_415.joblib", "mlp_prep.joblib", "mlp_415.keras"]:
        files.download(name)